# Ingeniería de Variables (Feature Engineering): El Problema Espacial

**Objetivo:** La regresión lineal clásica fracasa al interpretar coordenadas geográficas (latitud y longitud) porque asume relaciones monótonas en un plano rígido, ignorando los "epicentros" de riqueza en California (ej. San Francisco, Los Ángeles). 

En este notebook:
1. Engañar a la linealidad mediante **Feature Crosses** (cruces condicionales).
2. Transformar el espacio continuo en zonas discretas mediante **Spatial Binning (K-Means)**.

El resultado final será una matriz de datos enriquecida, lista para ser devorada por el modelo lineal.

### Inicialización y Carga

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

#cargar datos limpios del analisis EDA anterior
df = pd.read_csv('../data/raw/housing_clean.csv')
print(f"Base cargada. Dimensiones iniciales: {df.shape}")

Base cargada. Dimensiones iniciales: (19087, 13)


## Interacciones Condicionales (Feature Crosses)
La matriz de Pearson es ciega a los contextos. Una casa antigua pierde valor, *a menos* que esté en un barrio de altos ingresos (casa histórica). Un precio sube por el tamaño, *a menos* que haya demasiadas personas viviendo ahí (hacinamiento). 

Inyectar estas dos nuevas dimensiones matemáticas multiplicando y dividiendo variables existentes para que el modelo lineal asigne pesos a estos contextos específicos.

In [3]:
#interaccion edad-ingreso
df['age_x_income'] = df['housing_median_age'] * df['median_income']

#ratio de hacinamiento
df['population_per_room'] = df['population'] / df['total_rooms']

print("Feature Crosses inyectados en la matriz.")

Feature Crosses inyectados en la matriz.


## Discretización del Espacio (Spatial Binning con K-Means)
La latitud y longitud puras no significan nada para el algoritmo predictivo. Utilizar aprendizaje no supervisado (K-Means Clustering) para leer *únicamente* las coordenadas y agrupar el estado de California en 10 macro-zonas o epicentros. 

Esto transforma un problema de "coordenadas infinitas" en una variable categórica finita que el modelo lineal puede usar como un "escalón" de precio.

In [4]:
#solo 10 epicentros
n_clusters = 10

#extraer coordenadas
coords = df[['latitude', 'longitude']]

#entrenar el agrupador espacial
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['geo_cluster'] = kmeans.fit_predict(coords)

#convertir el cluster en texto para que el modelo no asuma el valor por su clasificacion numerica
df['geo_cluster'] = df['geo_cluster'].astype(str) 

print(f"Mapa dividido en {n_clusters} zonas geográficas discretas.")

Mapa dividido en 10 zonas geográficas discretas.
